In [1]:
from abc import ABC, abstractmethod
import numpy as np
from dataclasses import dataclass

In [2]:
class IShip(ABC):
    """
    Interface for controllable spacecraft.
    """

    @abstractmethod
    def reset(self, x, v, a):
        """
        Called on environment reset.
        Can initialize internal state.

        x: array of position vectors for every body in sim
        v: array of velocity vectors for every body in sim
        a: array of acceleration vectors for every body in sim
        """
        pass

    @abstractmethod
    def control_acceleration(self, x, v, a, action):
        """
        Returns acceleration vector to be added to physics.
        Shape: (2,) or (dim,)

        x: array of position vectors for every body in sim
        v: array of velocity vectors for every body in sim
        a: array of acceleration vectors for every body in sim
        """
        pass

    @abstractmethod
    def observe(self, x, v, a):
        """
        Returns observation vector for RL.

        x: array of position vectors for every body in sim
        v: array of velocity vectors for every body in sim
        a: array of acceleration vectors for every body in sim
        """
        pass

    @abstractmethod
    def reward(self, x, v, a):
        """
        Returns scalar reward.

        x: array of position vectors for every body in sim
        v: array of velocity vectors for every body in sim
        a: array of acceleration vectors for every body in sim
        """
        pass

    @abstractmethod
    def done(self,  x, v, a):
        """
        Returns termination flag.

        x: array of position vectors for every body in sim
        v: array of velocity vectors for every body in sim
        a: array of acceleration vectors for every body in sim
        """
        pass

    @abstractmethod
    def done(self,  x, v, a, max_steps, step_count):
        """
        Returns termination flag.

        x: array of position vectors for every body in sim
        v: array of velocity vectors for every body in sim
        a: array of acceleration vectors for every body in sim
        """
        pass

In [3]:
class Actions:
    def __init__(self, actions: dict[int, np.ndarray]):
        """
        actions: {action_id: direction_vector}
        """
        self.actions = actions
        self.n = len(actions)

    def sample(self):
        return np.random.randint(0, self.n)

    def vector(self, action: int):
        return self.actions[action]

In [4]:
@dataclass
class Maneuver:
    start: float        # seconds
    duration: float     # seconds
    action: int         # discrete action (from Actions)

def action_from_maneuvers(t: float, maneuvers: list[Maneuver]) -> int | None:
    for m in maneuvers:
        if m.start <= t < m.start + m.duration:
            return m.action
    return None

In [5]:
simple_action_space = Actions({
    0: np.array([0.0, 0.0]),
    1: np.array([1.0, 0.0]),
    2: np.array([-1.0, 0.0]),
    3: np.array([0.0, 1.0]),
    4: np.array([0.0, -1.0]),
})

In [6]:
class SimpleImpulseShip(IShip):
    def __init__(self,
                 ship_index: int, mass: float, thrust: float,
                 actions: Actions, safety_radius: float | None = None, escape_dist: float | None = None,  escape_vel: float | None = None,
                 target_dist: float | None = None, target_vel: float | None = None, target_index: int | None = None,
                 pos_scale: float | None = None, vel_scale: float | None = None, reward_coef: float | None = None):
        # Spacecraft should always have last index
        self.ship_index = ship_index
        self.mass = mass if mass > 0 else 0.00000001
        self.actions = actions
        self.done_flag = False

        self.last_action = None
        self.last_dx = None
        self.last_vx = None
        self.prev_dist_norm = None
        self.prev_vel_norm = None

        if escape_dist is not None:
            self.escape_dist = escape_dist if escape_dist > 0 else 1e9
        else:
            self.escape_dist = 1e9
        if escape_vel is not None:
            self.escape_vel = escape_vel if escape_vel > 0 else 1e5
        else:
            self.escape_vel = 1e5
        
        self.target_index = target_index
        if target_dist is not None:
            self.target_dist = target_dist if target_dist > 0 else 0.1
        if target_vel is not None:
            self.target_vel = target_vel if target_vel > 0 else 0.1
        

        self.thrust = thrust if thrust >= 0 else 0
        if safety_radius is not None:
            self.safety_radius = safety_radius if safety_radius >= 0 else 0

        # make sure pos scale is never 0
        if pos_scale is not None:
            self.pos_scale = pos_scale
        elif safety_radius is not None and safety_radius > 0:
            self.pos_scale = safety_radius
        else:
            self.pos_scale = 0.1

        # make sure vel scale is never 0
        if vel_scale is not None:
            self.vel_scale = vel_scale
        else:
            self.vel_scale = thrust / mass if thrust > 0 else 0.1

        if reward_coef is not None:
            self.reward_coef = reward_coef
        else:
            self.reward_coef = 1.0
        
    
    def reset(self, x, v, a):
        self.done_flag = False
        self.last_action = None
        self.last_dx = None
        self.last_vx = None
        self.prev_dist_norm = None
        self.prev_vel_norm = None

    
    def control_acceleration(self, x, v, a, action: int):
        self.last_action = action
        direction = self.actions.vector(action)
        return (self.thrust / self.mass) * direction

    def observe(self, x, v, a):
        ship_x = x[self.ship_index]
        ship_v = v[self.ship_index]
    
        obs = []

        # absolute state (scaled) not that usefull for chasing an object on a complex trajectory
        #obs.append(ship_x / self.pos_scale)
        #obs.append(ship_v / self.vel_scale)
    
        if self.target_index is not None:
            dx = ship_x - x[self.target_index]
            dv = ship_v - v[self.target_index]

            dist = np.linalg.norm(dx)
            speed = np.linalg.norm(dv)
    
            obs.append(dx / self.pos_scale)
            obs.append(dv / self.vel_scale)
            obs.append(np.array([dist / self.pos_scale]))
            obs.append(np.array([speed / self.vel_scale]))
    
        return np.concatenate(obs)

    def reward(self, x, v, a):
        if self.target_index is None:
            return 0.0
    
        dx = np.linalg.norm(x[self.ship_index] - x[self.target_index]) # relative distance vector
        dv = np.linalg.norm(v[self.ship_index] - v[self.target_index]) # relative velocity vector

        # normilise vectors
        dist_norm = dx / self.target_dist 
        vel_norm  = dv / self.target_vel
    
        # initialize memory
        if self.prev_dist_norm is None:
            self.prev_dist_norm = dist_norm
        if self.prev_vel_norm is None:
            self.prev_vel_norm = vel_norm
    
        #self.reward_coef: total episode reward magnitude ≈ O(1)  so 1.0 / self.max_steps
    
        r = 0.0
    
        # 1. time pressure (very small, always on)
        r -= 0.2 * self.reward_coef
    
        # 2. distance progress (PRIMARY objective)
        dist_progress = self.prev_dist_norm - dist_norm
        r += 20.0 * dist_progress * self.reward_coef
        self.prev_dist_norm = dist_norm
    
        # 3. velocity shaping (ONLY when close)
        if dist_norm < 3.0:
            vel_progress = self.prev_vel_norm - vel_norm
            r += 0.5 * vel_progress * self.reward_coef
        self.prev_vel_norm = vel_norm
    
        # 4. thrust / acceleration penalty (weak)
        #if self.last_action != 0:
        #    r -= 0.1 * self.reward_coef
    
        # 5. terminal conditions
        # success
        if dist_norm <= 1.0 and vel_norm <= 1.0:
            r += 10.0
            self.done_flag = True
            print("Success: Ship reached the target")
            return r
    
        # crash
        dist_to_primary = np.linalg.norm(x[self.ship_index] - x[0])
        if dist_to_primary < self.safety_radius:
            r -= 1.0
            self.done_flag = True
            print("Failure: Ship crashed")
            return r
    
        # escape (distance OR velocity)
        vel_rel = np.linalg.norm(v[self.ship_index] - v[0])
        if dist_to_primary > self.escape_dist or vel_rel > self.escape_vel:
            r -= 1.0
            self.done_flag = True
            print("Failure: Ship escaped the system or exceeded system escape velocity")
            return r
    
        return r
    
    def done(self, x, v, a):
        return self.done_flag